# install

In [ ]:
! pip install lightning
! pip install torchview
! pip install torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.9/827.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 31.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-c

# import

In [ ]:
# ===============================
# PyTorch Core Modules
# ===============================
import torch  # Core tensor operations
from torch import Tensor
import torch.nn as nn  # Neural network layers
import torch.nn.functional as F  # Functional API for activations, losses, etc.
import torch.optim as optim  # Optimizers
from torch.utils.data import Dataset, DataLoader, random_split  # Dataset utilities

# ===============================
# PyTorch Ecosystem
# ===============================
import pytorch_lightning as pl  # High-level training framework
import torchmetrics  # Model evaluation metrics
from torchview import draw_graph  # Model visualization
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor # Callbacks

# ===============================
# TorchVision - For image data
# ===============================
import torchvision  # Pretrained models and datasets
import torchvision.transforms as transforms  # Image preprocessing utilities
import torchvision.models as models
from torchvision.io import read_image, decode_image
from torchvision.models import resnet50, ResNet50_Weights, MobileNet_V3_Large_Weights

# ===============================
# Scikit-learn - Data generation and preprocessing
# ===============================
from sklearn.datasets import make_regression  # Synthetic regression data
from sklearn.model_selection import train_test_split  # Train/test splitting
from sklearn.preprocessing import MinMaxScaler  # Feature scaling

# ===============================
# Visualization
# ===============================
import matplotlib.pyplot as plt  # Plotting utilities

# ===============================
# Numerical Computing
# ===============================
import numpy as np  # Array operations
import math  # Basic mathematical functions

from PIL import Image
import requests
import io
import json

import os

# Dataset

## Load

In [ ]:
import kagglehub

path = kagglehub.dataset_download("yudhaislamisulistya/plants-type-datasets")

print("Path to dataset files:", path)
# %ls -R {path}

Path to dataset files: /kaggle/input/


## PlantDataset


In [ ]:
from torchvision.io import read_image

class PlantDataset(Dataset):
    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.image_files = []
        self.labels = []
        # []
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                idx = self.class_to_idx[class_name]
                for image_file in os.listdir(class_dir):
                    if image_file.endswith(('.jpg', '.png')):
                        self.image_files.append(os.path.join(class_dir, image_file))
                        self.labels.append(idx)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = read_image(img_path)
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

## transforms



In [ ]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ConvertImageDtype(torch.float), # Add this line to convert to float
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((32,32)), # Changed from 128x128 to 32x32
    transforms.ConvertImageDtype(torch.float), # Add this line to convert to float
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((32,32)), # Changed from
    transforms.ConvertImageDtype(torch.float), # Add this line to convert to float
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## DataLoader


In [ ]:
train_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Train_Set_Folder'
val_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Validation_Set_Folder'
test_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Test_Set_Folder'

classes = sorted(os.listdir(train_root_dir))
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

train_dataset = PlantDataset(
    root_dir=train_root_dir,
    transform=train_transforms,
    class_to_idx=class_to_idx
    )
val_dataset = PlantDataset(
    root_dir=val_root_dir,
    transform=val_transforms,
    class_to_idx=class_to_idx
    )
test_dataset = PlantDataset(
    root_dir=test_root_dir,
    transform=test_transforms,
    class_to_idx=class_to_idx
    )

batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in training DataLoader: {len(train_dataloader)}")
print(f"Number of batches in validation DataLoader: {len(val_dataloader)}")
print(f"Number of batches in test DataLoader: {len(test_dataloader)}")

Number of batches in training DataLoader: 375
Number of batches in validation DataLoader: 48
Number of batches in test DataLoader: 47


# efficientnet b1

## wrapper

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchmetrics
import pytorch_lightning as pl
from torchvision.models import EfficientNet_B1_Weights

class EfficientNetB1PlantClassifierLightningModule(pl.LightningModule):
    def __init__(self, num_classes, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        weights = EfficientNet_B1_Weights.DEFAULT
        self.model = torchvision.models.efficientnet_b1(weights=weights)

        # for param in self.model.features.parameters():
        #     param.requires_grad = False

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

        self.criterion = nn.CrossEntropyLoss()

        self.train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

        self.test_precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.test_recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.test_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average='macro')


    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', self.train_accuracy(outputs, labels), prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', self.val_accuracy(outputs, labels), prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)

        # Update metrics (accumulate predictions)
        self.test_accuracy.update(outputs, labels)
        self.test_precision.update(outputs, labels)
        self.test_recall.update(outputs, labels)
        self.test_f1.update(outputs, labels)

        self.log('test_loss', loss)
        return loss

    def on_test_epoch_end(self):
        # Compute final metrics after all batches
        self.log('test_acc', self.test_accuracy.compute())
        self.log('test_precision', self.test_precision.compute())
        self.log('test_recall', self.test_recall.compute())
        self.log('test_f1', self.test_f1.compute())


    def configure_optimizers(self):
        # optimizer = optim.Adam(filter(lambda p: p.requires_grad, self.parameters()), lr=self.hparams.learning_rate)
        optimizer = optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return {"optimizer": optimizer}

    # def on_train_epoch_start(self):
    #     if self.current_epoch == 1:
    #         for param in self.model.features.parameters():
    #             param.requires_grad = True
    #         self.trainer.optimizers[0] = optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    #         print(f"Epoch {self.current_epoch}: Unfreezing EfficientNetB1 features and reconfiguring optimizer.")


num_classes = len(train_dataset.class_to_idx)
efficientnet_b1_lightning_model = EfficientNetB1PlantClassifierLightningModule(num_classes=num_classes)

print("✅ EfficientNet B1 Plant Classifier defined.")
print(efficientnet_b1_lightning_model)

Downloading: "https://download.pytorch.org/models/efficientnet_b1-c27df63c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1-c27df63c.pth
100%|██████████| 30.1M/30.1M [00:00<00:00, 161MB/s]


✅ EfficientNet B1 Plant Classifier defined.
EfficientNetB1PlantClassifierLightningModule(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1

## train

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    dirpath='checkpoints/',
    filename='best-efficientnet-b1-model',
    save_top_k=1,
    verbose=True
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=3,
    verbose=True
)

lr_monitor_callback = LearningRateMonitor(logging_interval='step')

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks.progress import TQDMProgressBar

# Make sure the bar updates properly in Kaggle
progress_bar = TQDMProgressBar(refresh_rate=1)

# Optional: enable detailed logs
logger = CSVLogger(save_dir="logs/", name="efficientnet_b0")

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stopping_callback, lr_monitor_callback, progress_bar],
    logger=logger,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=1
)

print("Starting EfficientNet B1 Plant Dataset training with callbacks...")
trainer.fit(efficientnet_b1_lightning_model, train_dataloader, val_dataloader)

print("\nStarting EfficientNet B1 Plant Dataset testing...")

test_results_b1 = trainer.test(efficientnet_b1_lightning_model, test_dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Starting EfficientNet B1 Plant Dataset training with callbacks...



  | Name           | Type                | Params | Mode 
---------------------------------------------------------------
0 | model          | EfficientNet        | 6.6 M  | train
1 | criterion      | CrossEntropyLoss    | 0      | train
2 | train_accuracy | MulticlassAccuracy  | 0      | train
3 | val_accuracy   | MulticlassAccuracy  | 0      | train
4 | test_accuracy  | MulticlassAccuracy  | 0      | train
5 | test_precision | MulticlassPrecision | 0      | train
6 | test_recall    | MulticlassRecall    | 0      | train
7 | test_f1        | MulticlassF1Score   | 0      | train
---------------------------------------------------------------
6.6 M     Trainable params
0         Non-trainable params
6.6 M     Total params
26.206    Total estimated model params size (MB)
480       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


...

Monitored metric val_loss did not improve in the last 3 records. Best score: 0.246. Signaling Trainer to stop.
Epoch 8, global step 3375: 'val_acc' was not in top 1



Starting EfficientNet B1 Plant Dataset testing...


/usr/local/lib/python3.11/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.8829219341278076     │
│          test_f1          │    0.8852807283401489     │
│         test_loss         │    0.3901823163032532     │
│      test_precision       │    0.8942397832870483     │
│        test_recall        │    0.8829360604286194     │
└───────────────────────────┴───────────────────────────┘